# 03.04 · Long-term memory with stores

**Module 03 · Memory & Context Engineering** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 150–180 min |
| 📋 Prerequisites | Lessons 03.01–03.03; Lesson 02.04 (`ToolRuntime`, `context_schema`). |
| 🔑 Keys | `GROQ_API_KEY` (optional). Optional local embeddings: `fastembed` (no key) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Use** the store API: `put`, `get`, `search`, `delete`, `list_namespaces`, and the `Item` it returns.
2. **Design** namespaces so users can't see each other's memories, and know the prefix-search trap.
3. **Read and write** user facts from tools with `runtime.store`, with the user id from `runtime.context`.
4. **Search** memories by meaning with an embedding `index` (offline hashing embeddings, then a real local model).
5. **Persist** the store in SQLite (with vector search and TTLs).

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Long-term memory](https://docs.langchain.com/oss/python/langchain/long-term-memory) | **Main page.** Store in `create_agent`, read/write from tools |
| [Stores](https://docs.langchain.com/oss/python/langgraph/stores) | `put`/`get`/`search`, paging, namespaces, semantic search `index`, `fields`, `index=False` |
| [Store integrations](https://docs.langchain.com/oss/python/integrations/long-term-memory/index) | SQLite, Postgres, Redis and other back ends |
| [FastEmbed](https://qdrant.github.io/fastembed/) | The local ONNX embedding library used for the optional real-embeddings part |

---
## Step 0 · Install and set up

The optional real-embeddings part needs `fastembed` (≈150 MB of wheels, no PyTorch). Install it once from **Git Bash** at the repo root, course env active:
```bash
python -m pip install -e ".[embeddings]"
```
Without it, that part prints `⏭️ Skipped` and everything else runs.

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2" "langgraph-checkpoint-sqlite>=3.1.1,<4"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langgraph", "langgraph-checkpoint-sqlite",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/03-memory-and-context/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence one known, harmless warning (same as Lesson 02.04).
# langchain-core 1.6.9 and 1.6.10 serialize the injected ToolRuntime/Runtime while parsing input and Pydantic
# warns "PydanticSerializationUnexpectedValue ... field_name='context'". Context still arrives intact.
import warnings

warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

---
## Part 1 · The store API

A store is a key-value database of **JSON objects** organised in **namespaces** (tuples of strings, like folders):

| Call | Does |
|---|---|
| `store.put(namespace, key, value)` | Create or **overwrite** `value` (a dict) |
| `store.get(namespace, key)` | One `Item` or `None` |
| `store.search(namespace_prefix, query=None, filter=None, limit=10, offset=0)` | Items under a **prefix**; with `query=` and an index, ranked by similarity |
| `store.delete(namespace, key)` | Remove one item |
| `store.list_namespaces(prefix=..., max_depth=...)` | Which namespaces exist |

An `Item` has `namespace`, `key`, `value`, `created_at`, `updated_at` (UTC); a search hit (`SearchItem`) adds `score`.

In [ ]:
# 🧪 Offline: basic operations on an in-memory store.
from mastery.memory import learner_namespace, make_store

store = make_store("memory")
ana = learner_namespace("ana")                       # ("learners", "ana")
store.put(ana, "profile", {"name": "Ana", "grade": 9, "style": "examples first"})
store.put(ana, "weak-fractions", {"topic": "fractions", "note": "mixes up numerator and denominator"})
store.put(learner_namespace("bruno"), "profile", {"name": "Bruno", "grade": 10, "style": "rules first"})

item = store.get(ana, "profile")
print("get    :", item.key, item.value, "| updated", item.updated_at.isoformat(timespec="seconds"))
print("search :", [i.key for i in store.search(ana)])
print("filter :", [i.key for i in store.search(ana, filter={"topic": "fractions"})])
store.delete(ana, "weak-fractions")
print("deleted:", store.get(ana, "weak-fractions"))
print("spaces :", store.list_namespaces(prefix=("learners",)))

**You should now see** Ana's profile, `search : ['profile', 'weak-fractions']`, `filter : ['weak-fractions']`, `deleted: None`, and two namespaces.

---
## Part 2 · Namespace design (and the prefix trap)

`search(prefix)` returns **everything below the prefix**. That's handy and dangerous:

In [ ]:
# 🧪 Offline: a too-short prefix returns every learner's data.
print("('learners',)        →", [(tuple(i.namespace), i.key) for i in store.search(("learners",))])
print("('learners', 'ana')  →", [(tuple(i.namespace), i.key) for i in store.search(ana)])

**You should now see** that the first search returns Bruno's profile **and** Ana's. If a tool ever builds the prefix from untrusted input (or forgets the user id), it leaks other users' memories.

Rules the course follows:

1. **Every user-scoped namespace starts with `("learners", user_id)`.** Built by one function, `learner_namespace(user_id, ...)`, so tools can't get it wrong.
2. **`user_id` comes from `runtime.context`,** set by your app, never from a tool argument (Lesson 02.04).
3. **Validate labels.** LangGraph rejects empty labels and labels with `.`; `safe_label` also rejects `*`, `/`, `..` and the like.
4. **Sub-namespaces for kinds of memory:** `("learners", id)` for the profile, `("learners", id, "facts")` for a collection, `("learners", id, "episodes")` for experiences.
5. **App-wide memories** (procedural, shared by everyone) go elsewhere, e.g. `("app", "instructions")`.

In [ ]:
# 🧪 Offline: bad ids fail loudly instead of reaching the store.
for bad in ("ana.perez", "", "*", "ana/../bruno"):
    try:
        learner_namespace(bad)
    except ValueError as error:
        print(f"{bad!r:<16} → {str(error)[:60]}…")

**You should now see** four `invalid namespace label` errors.

---
## Part 3 · Memory tools: `remember_fact` and `recall_facts`

Pass the store to `create_agent(..., store=store)`. Every tool then gets it as `runtime.store`, next to `runtime.context` (who is asking). The model chooses **what** to remember; your code chooses **where**.

In [ ]:
# 🧪 Offline: two memory tools. The model never sees or picks the user id.
import uuid
from dataclasses import dataclass
from typing import Literal

from langchain.tools import ToolRuntime, tool


@dataclass
class LearnerContext:
    user_id: str


Kind = Literal["profile", "weak_topic", "style", "other"]


@tool(parse_docstring=True)
def remember_fact(fact: str, kind: Kind, runtime: ToolRuntime[LearnerContext]) -> str:
    """Save one short, durable fact about the current learner for future sessions.

    Args:
        fact: The fact in one sentence, e.g. 'Struggles with dividing fractions'.
        kind: profile (who they are), weak_topic, style (how they like explanations), or other.
    """
    ns = learner_namespace(runtime.context.user_id, "facts")
    runtime.store.put(ns, str(uuid.uuid4()), {"fact": fact[:300], "kind": kind})
    return f"Saved ({kind}): {fact[:300]}"


@tool(parse_docstring=True)
def recall_facts(query: str, runtime: ToolRuntime[LearnerContext]) -> str:
    """Look up saved facts about the current learner.

    Args:
        query: What you want to know, e.g. 'weak topics' or 'preferred explanation style'.
    """
    hits = runtime.store.search(learner_namespace(runtime.context.user_id, "facts"), query=query, limit=5)
    return "\n".join(f"- ({h.value['kind']}) {h.value['fact']}" for h in hits) or "No saved facts yet."

In [ ]:
# 🧪 Offline: session 1 saves a fact; session 2 (new thread!) recalls it; Bruno sees nothing.
from langchain.agents import create_agent
from langchain.messages import AIMessage
from mastery.agents import format_transcript
from mastery.memory import make_checkpointer, thread_config
from mastery.testing import ScriptedChatModel

facts_store = make_store("memory")
checkpointer = make_checkpointer("memory")


def memory_agent(responses):
    return create_agent(ScriptedChatModel(responses=responses), tools=[remember_fact, recall_facts],
                        context_schema=LearnerContext, store=facts_store, checkpointer=checkpointer)


session1 = memory_agent([
    AIMessage("", tool_calls=[{"name": "remember_fact", "id": "m1",
                               "args": {"fact": "Struggles with dividing fractions", "kind": "weak_topic"}}]),
    "Noted! We'll practise dividing fractions.",
])
session1.invoke({"messages": [{"role": "user", "content": "Dividing fractions is so hard for me."}]},
                thread_config("ana-1"), context=LearnerContext("ana"))

recall_script = [AIMessage("", tool_calls=[{"name": "recall_facts", "id": "r1", "args": {"query": "weak topics"}}]),
                 lambda msgs: f"From last time: {msgs[-1].content}"]
for user, thread in (("ana", "ana-2"), ("bruno", "bruno-1")):
    out = memory_agent(list(recall_script)).invoke(
        {"messages": [{"role": "user", "content": "What should we practise today?"}]},
        thread_config(thread), context=LearnerContext(user))
    print(f"{user}: {out['messages'][-1].content}")

**You should now see** `ana: From last time: - (weak_topic) Struggles with dividing fractions` and `bruno: From last time: No saved facts yet.`

### What just happened?

- Session 2 used a **new thread** (`ana-2`): no messages from session 1 were loaded. The fact came from the **store**, which is shared by all threads.
- Both tools built the namespace from `runtime.context.user_id`. The same scripted model output (`recall_facts(query='weak topics')`) gave Bruno nothing, because his namespace is empty. Isolation comes from the code, not from the prompt.
- The store has no index yet, so `query=` is ignored and search returns items in insertion order. That's fine for 3 facts, not for 300. Part 4 fixes it.

In [ ]:
# 💸 Live: does a real model decide to remember? (2–4 short calls)
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    live_agent = create_agent(
        get_model("fast", temperature=0, reasoning_effort="low"), tools=[remember_fact, recall_facts],
        context_schema=LearnerContext, store=facts_store, checkpointer=checkpointer,
        system_prompt=("You are a patient math tutor. When the learner tells you something durable about "
                       "themselves (weak topics, how they like explanations), save it with remember_fact. "
                       "Keep replies to two sentences."))
    out = live_agent.invoke(
        {"messages": [{"role": "user", "content": "I learn best with drawings, and percentages confuse me."}]},
        {**thread_config("ana-live"), "recursion_limit": 10}, context=LearnerContext("ana"))
    print(format_transcript(out["messages"]))
    print("\nAna's facts now:", [i.value["fact"] for i in facts_store.search(learner_namespace("ana", "facts"))])
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** one or two `remember_fact` calls (a `style` fact about drawings, a `weak_topic` about percentages) and Ana's fact list growing. Models differ: if it saves nothing, make the system prompt more explicit. That is *procedural* memory at work.

---
## Part 4 · Semantic search: an embedding index

Give the store an `index` and `put` also stores an **embedding** of the chosen `fields`; `search(..., query=...)` ranks by similarity:

```python
make_store("memory", embed=embeddings, dims=..., fields=["fact"])
# same as InMemoryStore(index={"embed": embeddings, "dims": ..., "fields": ["fact"]})
```

`fields=["$"]` (the default) embeds the whole value. `store.put(..., index=False)` skips embedding for one item (it's still retrievable, just not by `query`).

Offline, `mastery.memory.HashingEmbeddings` stands in: deterministic and free, but it matches **words**, not meaning. Watch where it fails.

In [ ]:
# 🧪 Offline: hashing embeddings find shared words, and miss synonyms.
from mastery.memory import HashingEmbeddings

FACTS = [
    ("weak_topic", "Struggles with dividing fractions"),
    ("style", "Prefers drawings and visual explanations"),
    ("profile", "Is preparing for the ICFES exam in November"),
    ("weak_topic", "Confuses percentages with decimals"),
    ("other", "Loves football and plays on Saturdays"),
]
QUERIES = ["fractions division", "how do they like things explained?", "upcoming test", "sports"]


def build(embeddings, dims=None):
    s = make_store("memory", embed=embeddings, dims=dims, fields=["fact"])
    for i, (kind, fact) in enumerate(FACTS):
        s.put(learner_namespace("ana", "facts"), f"f{i}", {"fact": fact, "kind": kind})
    return s


def show(s):
    for q in QUERIES:
        top = s.search(learner_namespace("ana", "facts"), query=q, limit=1)[0]
        print(f"{q:<36} → {top.value['fact']:<45} (score {top.score:.2f})")


show(build(HashingEmbeddings()))

**You should now see** the first query correct (score ≈ 0.41: it shares *fraction* and *divid…* with the fact) and the other three wrong. *"upcoming test"* and *"sports"* score **0.00**: no shared word with any fact, so the "top" hit is just the first item. Word overlap can't know that a test is an exam or that football is a sport.

In [ ]:
# 🌐 Optional: a real local embedding model (first run downloads ~70 MB to ~/.cache/fastembed).
import importlib.util

HAVE_FASTEMBED = importlib.util.find_spec("fastembed") is not None
if HAVE_FASTEMBED:
    from mastery.memory import local_embeddings

    local = local_embeddings()          # BAAI/bge-small-en-v1.5, 384 dimensions, runs on CPU
    print("model:", local.model_name, "| dims:", local.dims)
    show(build(local))
else:
    print('⏭️ Skipped: fastembed is not installed. Git Bash, repo root: python -m pip install -e ".[embeddings]"')

**You should now see** (with `fastembed`) all four queries matched to the right fact, including *"upcoming test" → ICFES exam* and *"sports" → football*, because the model embeds meaning. First run: a short download progress bar. Later runs work **offline**.

### What just happened?

- `put` called `embed_documents` on the `fact` field and stored the vector. `search(query=...)` called `embed_query`, then ranked items by cosine similarity (`score`).
- Same store API, same tools: only the `index` changed. That's why `recall_facts` passed `query=` from the start.
- Scores aren't probabilities. Compare them between results of one query, don't threshold them blindly across models.

---
## Part 5 · A persistent store: SQLite (with vector search and TTL)

`make_store("sqlite", path, ...)` returns a `SqliteStore`. With an index it loads the **sqlite-vec** extension into SQLite (installed with `langgraph-checkpoint-sqlite`). `ttl={"default_ttl": minutes}` makes items expire; `store.sweep_ttl()` deletes expired ones.

In [ ]:
# 🧪 Offline: write with one store object, close it, read with a new one (as after a restart).
from mastery.memory import close, memory_dir

STORE_DB = memory_dir(REPO_ROOT) / "03-04-store.sqlite"
writer = make_store("sqlite", STORE_DB, embed=HashingEmbeddings(), fields=["fact"])
writer.put(learner_namespace("ana", "facts"), "exam", {"fact": "ICFES exam in November", "kind": "profile"})
close(writer)

reader = make_store("sqlite", STORE_DB, embed=HashingEmbeddings(), fields=["fact"])
hit = reader.search(learner_namespace("ana", "facts"), query="when is the exam", limit=1)[0]
print("after reopen:", hit.value, f"(score {hit.score:.2f})")
close(reader)

**You should now see** `after reopen: {'fact': 'ICFES exam in November', 'kind': 'profile'} (score …)`.

> **Same embeddings on write and read.** Vectors saved with one model are meaningless to another. If you switch `HashingEmbeddings` → `local_embeddings()`, use a new file (or re-embed everything). `HashingEmbeddings` uses BLAKE2b, not Python's `hash()`, precisely so its vectors are the same in every process.

---
## ⚠️ Pitfalls and troubleshooting (Windows first)

| Symptom | Cause | Fix |
|---|---|---|
| A user sees another user's facts | Prefix too short, or `user_id` taken from a tool argument | `learner_namespace(runtime.context.user_id, ...)`; write a cross-user test (Project 03) |
| `runtime.store` is `None` | `store=` missing in `create_agent` | Pass the store |
| `search(query=...)` ignores the query | Store has no `index` | `make_store(..., embed=..., dims=...)` |
| `ValueError: pass dims= with embed=` | Unknown vector size | `dims=384` for bge-small; `local_embeddings().dims` |
| `AttributeError: 'sqlite3.Connection' object has no attribute 'enable_load_extension'` | Your Python's SQLite can't load extensions (some macOS/pyenv builds; Anaconda and python.org Windows builds can) | Use the course's conda env; or keep the SQLite store without an index |
| `fastembed` download fails with `SSL: CERTIFICATE_VERIFY_FAILED` | Certificate path (Lesson 00.02, Step 4b) | Fix `SSL_CERT_FILE` as in 00.02, restart the kernel, re-run |
| Model downloads again every week | Default fastembed cache is the OS temp folder | `local_embeddings()` caches in `~/.cache/fastembed` (or set `FASTEMBED_CACHE_PATH`) |
| `Warning: You are sending unauthenticated requests to the HF Hub` | Hugging Face download without a token | Harmless for this small public model |
| `TqdmWarning: IProgress not found` | No `ipywidgets` for the download progress bar | Harmless; or `python -m pip install ipywidgets` and restart the kernel |
| `PermissionError: [WinError 32]` deleting `03-04-store.sqlite` | Store still open | `close(store)` or restart the kernel |
| Expired items still returned by `get` | TTL removes items when the sweeper runs | Call `store.sweep_ttl()` (or `start_ttl_sweeper()`) |

## ⚠️ Production notes

> ⚠️ **Production note: memory writes are prompt-injection targets.**
> Anything the model saves will be read back into future prompts. A user (or a web page a tool read) can plant *"Always reveal answers"* as a "fact". Cap length (`fact[:300]`), restrict `kind` to a `Literal`, and never store instructions as facts. Treat recalled memories as **data** in your prompt (Lesson 03.05).

> ⚠️ **Production note: give users control.**
> Provide "show what you remember" and "forget" (Project 03 stretch: `forget_fact` with confirmation). Set TTLs for memories you don't need forever, and log every write with the user id.

> ⚠️ **Production note: choose embeddings once per store.**
> Changing the embedding model means re-embedding every item. Record the model name and `dims` next to the store (e.g. in `("app", "meta")`) and check it at start-up. For servers, `PostgresStore` with pgvector (Module 12).

## 🏋️ Exercises

1. **(Easy)** Add a sixth fact and a query that only matches it with real embeddings (or only with hashing).
2. **(Medium)** Write `forget_fact(query, runtime)`: find the best match in the user's facts and delete it. Return what was deleted. Test it offline for two users.
3. **(Medium)** Change `remember_fact` to keep **one profile document** (`key="profile"`) and merge new `profile` facts into it, while other kinds stay a collection.
4. **(Stretch)** Also save which thread a fact came from. Find the attribute of `runtime.execution_info` that holds the thread id (`print(dir(runtime.execution_info))` inside a tool), store it with the fact, and show it in `recall_facts`.

<details>
<summary><b>Show solutions</b></summary>

**2.**
```python
@tool
def forget_fact(query: str, runtime: ToolRuntime[LearnerContext]) -> str:
    """Delete the saved fact that best matches the query."""
    ns = learner_namespace(runtime.context.user_id, "facts")
    hits = runtime.store.search(ns, query=query, limit=1)
    if not hits:
        return "Nothing to forget."
    runtime.store.delete(ns, hits[0].key)
    return f"Forgot: {hits[0].value['fact']}"
```
Project 03's stretch adds a confirmation step before the delete.

**4.** `runtime.execution_info.thread_id` (it also has `run_id`, `checkpoint_id`, `node_attempt`, ...). Store `{"fact": fact, "kind": kind, "source_thread": runtime.execution_info.thread_id}`.
</details>

## ✅ Checkpoint

- [ ] I can use `put`, `get`, `search`, `delete` and `list_namespaces`.
- [ ] I can explain the prefix-search trap and the course's namespace rules.
- [ ] I can write memory tools that take the user id from `runtime.context`.
- [ ] I can add an embedding index and explain why hashing embeddings miss synonyms.
- [ ] I can persist a store in SQLite and reopen it.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Open [Store integrations](https://docs.langchain.com/oss/python/integrations/long-term-memory/index). Which stores support **semantic search** and which support **TTL**? Then revisit your answer from Lesson 03.01's research checkpoint (the [feature comparison](https://docs.langchain.com/oss/python/concepts/products#feature-comparison) memory rows): would you change your pick for a 3-month learner relationship? Log it in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- A store holds JSON values under namespaces; it is shared across threads and written explicitly.
- Namespaces are your isolation boundary: build them in one function from `runtime.context`.
- An `index` (embeddings + dims + fields) turns `search(query=...)` into semantic search.
- `make_store("sqlite", ...)` persists memories, with sqlite-vec for vectors and optional TTLs.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 03.05 · Context engineering** → [`03-05-context-engineering.ipynb`](03-05-context-engineering.ipynb).

[Module 03 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)